# Training a microWakeWord Model

This notebook steps you through training a basic microWakeWord model. It is intended as a **starting point** for advanced users. You should use Python 3.10.

**The model generated will most likely not be usable for everyday use; it may be difficult to trigger or falsely activates too frequently. You will most likely have to experiment with many different settings to obtain a decent model!**

In the comment at the start of certain blocks, I note some specific settings to consider modifying.

This runs on Google Colab, but is extremely slow compared to training on a local GPU. If you must use Colab, be sure to Change the runtime type to a GPU. Even then, it still slow!

At the end of this notebook, you will be able to download a tflite file. To use this in ESPHome, you need to write a model manifest JSON file. See the [ESPHome documentation](https://esphome.io/components/micro_wake_word) for the details and the [model repo](https://github.com/esphome/micro-wake-word-models/tree/main/models/v2) for examples.

In [ ]:
# Installs microWakeWord. Restart the kernel after this is finished.

import platform

if platform.system() == "Darwin":
    %pip install 'git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version'

%pip install 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'

%pip install -e ..

In [ ]:
# ============================================================
# CELL 2 - Generate One Test Sample of the Custom Wake Word
# ============================================================

target_word = "hi tech panda"

import os
import sys
import subprocess
from IPython.display import Audio

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

piper_dir = os.path.abspath("piper-sample-generator")

model_dir = os.path.join(
    piper_dir,
    "models"
)

model_path = os.path.join(
    model_dir,
    "en_US-libritts_r-medium.pt"
)

output_dir = os.path.abspath("generated_samples")


# ------------------------------------------------------------
# Clone Piper Sample Generator
# ------------------------------------------------------------

if not os.path.exists(piper_dir):

    print("Cloning Piper Sample Generator...")

    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/rhasspy/piper-sample-generator",
            piper_dir
        ],
        check=True
    )


# ------------------------------------------------------------
# Install Required Packages
# ------------------------------------------------------------

print("Installing required packages...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "torch",
        "torchaudio",
        "piper-tts==1.3.0"
    ],
    check=True
)


# ------------------------------------------------------------
# Download Piper Voice Model
# ------------------------------------------------------------

os.makedirs(model_dir, exist_ok=True)

if not os.path.exists(model_path):

    print("Downloading Piper sample generation model...")

    subprocess.run(
        [
            "powershell",
            "-Command",
            f'Invoke-WebRequest '
            f'-Uri "https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt" '
            f'-OutFile "{model_path}"'
        ],
        check=True
    )


# ------------------------------------------------------------
# Generate One Test Sample
# ------------------------------------------------------------

os.makedirs(output_dir, exist_ok=True)

print(f'Generating test sample for: "{target_word}"')

subprocess.run(
    [
        sys.executable,
        "-m",
        "piper_sample_generator",
        target_word,
        "--model",
        model_path,
        "--max-samples",
        "1",
        "--batch-size",
        "1",
        "--output-dir",
        output_dir
    ],
    cwd=piper_dir,
    check=True
)


# ------------------------------------------------------------
# Play Generated Sample
# ------------------------------------------------------------

sample_file = os.path.join(output_dir, "0.wav")

if not os.path.exists(sample_file):
    raise FileNotFoundError(
        "The wake-word sample was not generated."
    )

print("Sample generated successfully!")

Audio(
    filename=sample_file,
    autoplay=True
)

In [ ]:
# Generates the full wake-word training sample set.

import os
import sys
import subprocess

output_dir = os.path.abspath("generated_samples")

print(f'Generating 1000 samples for: "{target_word}"')

subprocess.run(
    [
        sys.executable,
        "-m",
        "piper_sample_generator",
        target_word,
        "--model",
        model_path,
        "--max-samples",
        "1000",
        "--batch-size",
        "100",
        "--output-dir",
        output_dir
    ],
    cwd=piper_dir,
    check=True
)

print("1000 wake-word samples generated successfully.")

In [ ]:
# ============================================================
# ONE-GO CELL - Download and Prepare All Augmentation Datasets
# ============================================================

%pip install datasets==2.19.0 soundfile

import os
import zipfile
import tarfile
import requests
import datasets
import numpy as np
import scipy.io.wavfile
from pathlib import Path
from tqdm import tqdm


# ============================================================
# Helper: Download File
# ============================================================

def download_file(url, destination):

    # Remove empty/failed previous download
    if os.path.exists(destination) and os.path.getsize(destination) == 0:
        os.remove(destination)

    if os.path.exists(destination):
        print(f"Already downloaded: {destination}")
        return

    print(f"Downloading: {destination}")

    with requests.get(
        url,
        stream=True,
        allow_redirects=True
    ) as r:

        r.raise_for_status()

        total_size = int(
            r.headers.get("content-length", 0)
        )

        with open(destination, "wb") as f:

            with tqdm(
                total=total_size,
                unit="B",
                unit_scale=True,
                unit_divisor=1024
            ) as progress:

                for chunk in r.iter_content(
                    chunk_size=1024 * 1024
                ):

                    if chunk:
                        f.write(chunk)
                        progress.update(len(chunk))


# ============================================================
# 1. MIT Room Impulse Responses
# ============================================================

print("\n========================================")
print("STEP 1/3 - MIT Room Impulse Responses")
print("========================================")

mit_dir = "mit_rirs"

if not os.path.exists(mit_dir):

    os.makedirs(mit_dir, exist_ok=True)

    rir_dataset = datasets.load_dataset(
        "davidscripka/MIT_environmental_impulse_responses",
        split="train",
        streaming=True
    )

    for row in tqdm(
        rir_dataset,
        desc="Processing MIT RIR"
    ):

        name = Path(
            row["audio"]["path"]
        ).name

        output_file = os.path.join(
            mit_dir,
            name
        )

        scipy.io.wavfile.write(
            output_file,
            16000,
            (
                row["audio"]["array"] * 32767
            ).astype(np.int16)
        )

    print("MIT RIR dataset ready.")

else:

    print("MIT RIR dataset already exists. Skipping.")


# ============================================================
# 2. AudioSet
# ============================================================

print("\n========================================")
print("STEP 2/3 - AudioSet Background Audio")
print("========================================")

audioset_dir = "audioset"
audioset_16k_dir = "audioset_16k"

os.makedirs(
    audioset_dir,
    exist_ok=True
)

os.makedirs(
    audioset_16k_dir,
    exist_ok=True
)

audioset_archive = os.path.join(
    audioset_dir,
    "bal_train09.tar"
)

audioset_url = (
    "https://huggingface.co/datasets/"
    "agkphysics/AudioSet/resolve/"
    "196c0900867eff791b8f4d4be57db277e9a5b131/"
    "bal_train09.tar"
)

download_file(
    audioset_url,
    audioset_archive
)


# Extract AudioSet

audioset_audio_dir = os.path.join(
    audioset_dir,
    "audio"
)

if not os.path.exists(audioset_audio_dir):

    print("Extracting AudioSet...")

    with tarfile.open(
        audioset_archive,
        "r"
    ) as tar:

        tar.extractall(
            audioset_dir
        )

    print("AudioSet extracted.")

else:

    print("AudioSet already extracted.")


# Find FLAC files anywhere under extracted AudioSet

audioset_files = list(
    Path(audioset_dir).glob("**/*.flac")
)

print(
    f"Found {len(audioset_files)} AudioSet files."
)


# Convert AudioSet to 16 kHz WAV

if audioset_files:

    audioset_dataset = datasets.Dataset.from_dict({
        "audio": [
            str(file)
            for file in audioset_files
        ]
    })

    audioset_dataset = (
        audioset_dataset.cast_column(
            "audio",
            datasets.Audio(
                sampling_rate=16000
            )
        )
    )

    for row in tqdm(
        audioset_dataset,
        desc="Converting AudioSet"
    ):

        source_name = Path(
            row["audio"]["path"]
        ).stem

        output_file = os.path.join(
            audioset_16k_dir,
            source_name + ".wav"
        )

        if os.path.exists(output_file):
            continue

        audio_data = (
            row["audio"]["array"] * 32767
        ).astype(np.int16)

        scipy.io.wavfile.write(
            output_file,
            16000,
            audio_data
        )

    print("AudioSet conversion complete.")

else:

    raise RuntimeError(
        "AudioSet downloaded, but no FLAC files were found."
    )


# ============================================================
# 3. Free Music Archive
# ============================================================

print("\n========================================")
print("STEP 3/3 - Free Music Archive")
print("========================================")

fma_dir = "fma"
fma_16k_dir = "fma_16k"

os.makedirs(
    fma_dir,
    exist_ok=True
)

os.makedirs(
    fma_16k_dir,
    exist_ok=True
)

fma_archive = os.path.join(
    fma_dir,
    "fma_xs.zip"
)

fma_url = (
    "https://huggingface.co/datasets/"
    "mchl914/fma_xsmall/resolve/main/"
    "fma_xs.zip"
)

download_file(
    fma_url,
    fma_archive
)


# Extract FMA

fma_audio_dir = os.path.join(
    fma_dir,
    "fma_small"
)

if not os.path.exists(fma_audio_dir):

    print("Extracting Free Music Archive...")

    with zipfile.ZipFile(
        fma_archive,
        "r"
    ) as zip_ref:

        zip_ref.extractall(
            fma_dir
        )

    print("FMA extracted.")

else:

    print("FMA already extracted.")


# Find MP3 files

fma_files = list(
    Path(fma_dir).glob("**/*.mp3")
)

print(
    f"Found {len(fma_files)} FMA files."
)


# Convert FMA to 16 kHz WAV

if fma_files:

    fma_dataset = datasets.Dataset.from_dict({
        "audio": [
            str(file)
            for file in fma_files
        ]
    })

    fma_dataset = (
        fma_dataset.cast_column(
            "audio",
            datasets.Audio(
                sampling_rate=16000
            )
        )
    )

    for row in tqdm(
        fma_dataset,
        desc="Converting FMA"
    ):

        source_name = Path(
            row["audio"]["path"]
        ).stem

        output_file = os.path.join(
            fma_16k_dir,
            source_name + ".wav"
        )

        if os.path.exists(output_file):
            continue

        audio_data = (
            row["audio"]["array"] * 32767
        ).astype(np.int16)

        scipy.io.wavfile.write(
            output_file,
            16000,
            audio_data
        )

    print("FMA conversion complete.")

else:

    raise RuntimeError(
        "FMA downloaded, but no MP3 files were found."
    )


# ============================================================
# Final Verification
# ============================================================

mit_count = len(
    list(Path(mit_dir).glob("**/*.wav"))
)

audioset_count = len(
    list(Path(audioset_16k_dir).glob("*.wav"))
)

fma_count = len(
    list(Path(fma_16k_dir).glob("*.wav"))
)


print("\n========================================")
print("ALL AUGMENTATION DATASETS ARE READY")
print("========================================")

print(f"MIT RIR files     : {mit_count}")
print(f"AudioSet WAV files: {audioset_count}")
print(f"FMA WAV files     : {fma_count}")

In [ ]:
# Sets up the augmentations.
# To improve your model, experiment with these settings and use more sources of
# background clips.

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

clips = Clips(input_directory='generated_samples',
              file_pattern='*.wav',
              max_clip_duration_s=None,
              remove_silence=False,
              random_split_seed=10,
              split_count=0.1,
              )
augmenter = Augmentation(augmentation_duration_s=3.2,
                         augmentation_probabilities = {
                                "SevenBandParametricEQ": 0.1,
                                "TanhDistortion": 0.1,
                                "PitchShift": 0.1,
                                "BandStopFilter": 0.1,
                                "AddColorNoise": 0.1,
                                "AddBackgroundNoise": 0.75,
                                "Gain": 1.0,
                                "RIR": 0.5,
                            },
                         impulse_paths = ['mit_rirs'],
                         background_paths = ['fma_16k', 'audioset_16k'],
                         background_min_snr_db = -5,
                         background_max_snr_db = 10,
                         min_jitter_s = 0.195,
                         max_jitter_s = 0.205,
                         )


In [ ]:
# Augment a random clip and play it back to verify it works well

from IPython.display import Audio
from microwakeword.audio.audio_utils import save_clip

random_clip = clips.get_random_clip()
augmented_clip = augmenter.augment_clip(random_clip)
save_clip(augmented_clip, 'augmented_clip.wav')

Audio("augmented_clip.wav", autoplay=True)

In [ ]:
# Augment samples and save the training, validation, and testing sets.
# Validating and testing samples generated the same way can make the model
# benchmark better than it performs in real-word use. Use real samples or TTS
# samples generated with a different TTS engine to potentially get more accurate
# benchmarks.

import os
from mmap_ninja.ragged import RaggedMmap

output_dir = 'generated_augmented_features'

if not os.path.exists(output_dir):
    os.mkdir(output_dir)

splits = ["training", "validation", "testing"]
for split in splits:
  out_dir = os.path.join(output_dir, split)
  if not os.path.exists(out_dir):
      os.mkdir(out_dir)


  split_name = "train"
  repetition = 2

  spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenter,
                                     slide_frames=10,    # Uses the same spectrogram repeatedly, just shifted over by one frame. This simulates the streaming inferences while training/validating in nonstreaming mode.
                                     step_ms=10,
                                     )
  if split == "validation":
    split_name = "validation"
    repetition = 1
  elif split == "testing":
    split_name = "test"
    repetition = 1
    spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenter,
                                     slide_frames=1,    # The testing set uses the streaming version of the model, so no artificial repetition is necessary
                                     step_ms=10,
                                     )

  RaggedMmap.from_generator(
      out_dir=os.path.join(out_dir, 'wakeword_mmap'),
      sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=repetition),
      batch_size=100,
      verbose=True,
  )

In [ ]:
import os
import zipfile
import requests
from tqdm import tqdm

output_dir = './negative_datasets'
os.makedirs(output_dir, exist_ok=True)

link_root = (
    "https://huggingface.co/datasets/"
    "kahrendt/microwakeword/resolve/main/"
)

filenames = [
    'dinner_party.zip',
    'dinner_party_eval.zip',
    'no_speech.zip',
    'speech.zip'
]

for fname in filenames:

    link = link_root + fname
    zip_path = os.path.join(output_dir, fname)

    if not os.path.exists(zip_path):

        print(f"Downloading {fname}...")

        with requests.get(link, stream=True) as r:
            r.raise_for_status()

            total_size = int(
                r.headers.get('content-length', 0)
            )

            with open(zip_path, 'wb') as f, tqdm(
                total=total_size,
                unit='B',
                unit_scale=True,
                desc=fname
            ) as bar:

                for chunk in r.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if chunk:
                        f.write(chunk)
                        bar.update(len(chunk))

    print(f"Extracting {fname}...")

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(output_dir)

print("Negative datasets ready.")

In [ ]:
# Save a yaml config that controls the training process
# These hyperparamters can make a huge different in model quality.
# Experiment with sampling and penalty weights and increasing the number of
# training steps.

import yaml
import os

config = {}

config["window_step_ms"] = 10

config["train_dir"] = (
    "trained_models/wakeword"
)


# Each feature_dir should have at least one of the following folders with this structure:
#  training/
#    ragged_mmap_folders_ending_in_mmap
#  testing/
#    ragged_mmap_folders_ending_in_mmap
#  testing_ambient/
#    ragged_mmap_folders_ending_in_mmap
#  validation/
#    ragged_mmap_folders_ending_in_mmap
#  validation_ambient/
#    ragged_mmap_folders_ending_in_mmap
#
#  sampling_weight: Weight for choosing a spectrogram from this set in the batch
#  penalty_weight: Penalizing weight for incorrect predictions from this set
#  truth: Boolean whether this set has positive samples or negative samples
#  truncation_strategy = If spectrograms in the set are longer than necessary for training, how are they truncated
#       - random: choose a random portion of the entire spectrogram - useful for long negative samples
#       - truncate_start: remove the start of the spectrogram
#       - truncate_end: remove the end of the spectrogram
#       - split: Split the longer spectrogram into separate spectrograms offset by 100 ms. Only for ambient sets

config["features"] = [
    {
        "features_dir": "generated_augmented_features",
        "sampling_weight": 2.0,
        "penalty_weight": 1.0,
        "truth": True,
        "truncation_strategy": "truncate_start",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/speech",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/dinner_party",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/no_speech",
        "sampling_weight": 5.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    { # Only used for validation and testing
        "features_dir": "negative_datasets/dinner_party_eval",
        "sampling_weight": 0.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "split",
        "type": "mmap",
    },
]

# Number of training steps in each iteration - various other settings are configured as lists that corresponds to different steps
config["training_steps"] = [10000]

# Penalizing weight for incorrect class predictions - lists that correspond to training steps
config["positive_class_weight"] = [1]
config["negative_class_weight"] = [20]

config["learning_rates"] = [
    0.001,
]  # Learning rates for Adam optimizer - list that corresponds to training steps
config["batch_size"] = 128

config["time_mask_max_size"] = [
    0
]  # SpecAugment - list that corresponds to training steps
config["time_mask_count"] = [0]  # SpecAugment - list that corresponds to training steps
config["freq_mask_max_size"] = [
    0
]  # SpecAugment - list that corresponds to training steps
config["freq_mask_count"] = [0]  # SpecAugment - list that corresponds to training steps

config["eval_step_interval"] = (
    500  # Test the validation sets after every this many steps
)
config["clip_duration_ms"] = (
    1500  # Maximum length of wake word that the streaming model will accept
)

# The best model weights are chosen first by minimizing the specified minimization metric below the specified target_minimization
# Once the target has been met, it chooses the maximum of the maximization metric. Set 'minimization_metric' to None to only maximize
# Available metrics:
#   - "loss" - cross entropy error on validation set
#   - "accuracy" - accuracy of validation set
#   - "recall" - recall of validation set
#   - "precision" - precision of validation set
#   - "false_positive_rate" - false positive rate of validation set
#   - "false_negative_rate" - false negative rate of validation set
#   - "ambient_false_positives" - count of false positives from the split validation_ambient set
#   - "ambient_false_positives_per_hour" - estimated number of false positives per hour on the split validation_ambient set
config["target_minimization"] = 0.9
config["minimization_metric"] = None  # Set to None to disable

config["maximization_metric"] = "average_viable_recall"

with open(os.path.join("training_parameters.yaml"), "w") as file:
    documents = yaml.dump(config, file)

In [ ]:
# ============================================================
# TRAIN THE MICROWAKEWORD MODEL
# ============================================================

import sys
import subprocess

# Install TensorBoard required by the training script
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "tensorboard"
    ],
    check=True
)

print("\nStarting microWakeWord training...\n")

# Train the model
subprocess.run(
    [
        sys.executable,
        "-m",
        "microwakeword.model_train_eval",

        "--training_config=training_parameters.yaml",

        "--train", "1",
        "--restore_checkpoint", "1",

        "--test_tf_nonstreaming", "0",
        "--test_tflite_nonstreaming", "0",
        "--test_tflite_nonstreaming_quantized", "0",
        "--test_tflite_streaming", "0",
        "--test_tflite_streaming_quantized", "1",

        "--use_weights", "best_weights",

        "mixednet",

        "--pointwise_filters", "64,64,64,64",
        "--repeat_in_block", "1, 1, 1, 1",
        "--mixconv_kernel_sizes", "[5], [7,11], [9,15], [23]",
        "--residual_connection", "0,0,0,0",

        "--first_conv_filters", "32",
        "--first_conv_kernel_size", "5",
        "--stride", "3",
    ],
    check=True
)

print("\nTraining and model conversion completed.")

In [ ]:
# Downloads the tflite model file. To use on the device, you need to write a
# Model JSON file. See https://esphome.io/components/micro_wake_word for the
# documentation and
# https://github.com/esphome/micro-wake-word-models/tree/main/models/v2 for
# examples. Adjust the probability threshold based on the test results obtained
# after training is finished. You may also need to increase the Tensor arena
# model size if the model fails to load.

from google.colab import files

files.download(f"trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite")